# Notebook 1: Hybrid RAG with MongoDB `$rankFusion`

This notebook demonstrates:

- **Voyage AI** for embeddings
- **MongoDB Atlas `$rankFusion`** for native hybrid search fusion
- **Voyage AI `rerank-2.5`** for final re-ranking

> It is safe to run in GitHub Codespaces. If `VOYAGE_API_KEY` or `MONGODB_URI` is missing, external calls are skipped and the notebook still completes without errors.

In [ ]:
%pip install -q "voyageai" "pymongo" "langchain-mongodb" "python-dotenv" "cffi"

In [ ]:
import hashlib
import os
from dataclasses import dataclass
from typing import List

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


class VoyageAIEmbeddings(Embeddings):
    """LangChain-compatible embeddings wrapper for Voyage AI with a local fallback."""

    def __init__(self, api_key: str | None, model: str = "voyage-4", fallback_dim: int = 128):
        self.api_key = api_key
        self.model = model
        self.client = None
        self.fallback_dim = fallback_dim
        if api_key:
            try:
                import voyageai

                self.client = voyageai.Client(api_key=api_key)
            except Exception as e:
                print(f"Voyage AI unavailable, using local fallback embeddings: {e}")

    def _local_embed(self, text: str) -> List[float]:
        digest = hashlib.sha256(text.encode("utf-8")).digest()
        base = list(digest)
        return [((base[i % len(base)] / 255.0) * 2.0) - 1.0 for i in range(self.fallback_dim)]

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        if self.client:
            result = self.client.embed(texts, model=self.model, input_type="document")
            return result.embeddings
        return [self._local_embed(text) for text in texts]

    def embed_query(self, text: str) -> List[float]:
        if self.client:
            result = self.client.embed([text], model=self.model, input_type="query")
            return result.embeddings[0]
        return self._local_embed(text)


VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY")
MONGODB_URI = os.getenv("MONGODB_URI")
EMBED_MODEL = "voyage-4"
RERANK_MODEL = "rerank-2.5"
DB_NAME = "pyconlt2026"
COLLECTION_NAME = "rankfusion_demo"
VECTOR_INDEX_NAME = "vector_search_index"
TEXT_INDEX_NAME = "text_search_index"

embeddings = VoyageAIEmbeddings(VOYAGE_API_KEY, model=EMBED_MODEL)

print("VOYAGE_API_KEY set:", bool(VOYAGE_API_KEY))
print("MONGODB_URI set   :", bool(MONGODB_URI))
print("Voyage client active:", embeddings.client is not None)

In [ ]:
@dataclass
class DemoDoc:
    doc_id: str
    content: str

documents = [
    DemoDoc("doc-01", "Reciprocal Rank Fusion combines ranked lists without comparing raw scores."),
    DemoDoc("doc-02", "Relative Score Fusion works best when scores are normalized and calibrated."),
    DemoDoc("doc-03", "BM25 is a strong keyword retrieval method for exact terms, acronyms, and product codes."),
    DemoDoc("doc-04", "Dense vector retrieval finds semantically similar documents even when wording differs."),
    DemoDoc("doc-05", "Cross-encoder re-ranking scores each query-document pair directly."),
    DemoDoc("doc-06", "Hybrid search combines keyword retrieval and vector retrieval to improve recall and precision."),
    DemoDoc("doc-07", "RAG pipelines reduce hallucinations by grounding generation in retrieved context."),
    DemoDoc("doc-08", "Chunking strategy affects retrieval quality and downstream answer quality."),
    DemoDoc("doc-09", "Voyage AI provides embedding models such as voyage-4 for retrieval tasks."),
    DemoDoc("doc-10", "MongoDB Atlas Search supports full-text search and vector search in one platform."),
    DemoDoc("doc-11", "MongoDB `$rankFusion` applies native Reciprocal Rank Fusion across multiple pipelines."),
    DemoDoc("doc-12", "Re-ranking improves the final ordering after retrieval and fusion."),
]

print(f"Loaded {len(documents)} demo documents")
for doc in documents:
    print(f"- {doc.doc_id}: {doc.content}")

## Step 1: Generate embeddings with Voyage AI

If credentials are missing, the cell prints a message and skips the live API call.

In [ ]:
lc_documents = [
    Document(page_content=doc.content, metadata={"doc_id": doc.doc_id})
    for doc in documents
]

print(f"Prepared {len(lc_documents)} LangChain documents")
for doc in lc_documents:
    print(f"- {doc.metadata['doc_id']}: {doc.page_content}")

## Step 2: Insert data into MongoDB Atlas

This uses MongoDB Atlas as the storage layer for hybrid search.

In [ ]:
collection = None
vector_store = None

if not MONGODB_URI:
    print("Skipping MongoDB setup: MONGODB_URI is not set.")
else:
    from pymongo import MongoClient
    from langchain_mongodb import MongoDBAtlasVectorSearch

    client = MongoClient(MONGODB_URI)
    collection = client[DB_NAME][COLLECTION_NAME]
    collection.drop()

    vector_store = MongoDBAtlasVectorSearch(
        collection=collection,
        embedding=embeddings,
        index_name=VECTOR_INDEX_NAME,
        text_key="content",
        embedding_key="embedding",
        relevance_score_fn="cosine",
    )
    vector_store.add_documents(lc_documents)
    print(f"Inserted {len(lc_documents)} LangChain documents into {DB_NAME}.{COLLECTION_NAME}")

## Step 3: Create Atlas Search indexes (explicit Vector Search setup)

- Infer embedding dimension from Voyage output
- Create/update Atlas **Vector Search** index via `type="vectorSearch"`
- Create/update Atlas **Search** index via `type="search"`
- Wait for both indexes to reach `READY` state

In [ ]:
if collection is None or vector_store is None:
    print("Skipping index creation: MongoDB collection is not available.")
else:
    import time
    from pymongo.operations import SearchIndexModel

    # Infer embedding dimension from the actual model output to avoid Atlas dimension mismatch.
    sample_vector = embeddings.embed_query("dimension probe")
    vector_dimensions = len(sample_vector)
    print(f"Detected embedding dimensions: {vector_dimensions}")

    # Read existing Atlas Search / Vector Search indexes.
    try:
        existing_indexes = {idx["name"]: idx for idx in collection.list_search_indexes()}
    except Exception:
        existing_indexes = {}

    existing_vector_name = next(
        (name for name, idx in existing_indexes.items() if idx.get("type") == "vectorSearch"),
        None,
    )
    existing_text_name = next(
        (name for name, idx in existing_indexes.items() if idx.get("type") == "search"),
        None,
    )

    # Vector index via PyMongo for explicit Atlas Vector Search setup.
    vector_definition = {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": vector_dimensions,
                "similarity": "cosine",
            }
        ]
    }

    if VECTOR_INDEX_NAME in existing_indexes:
        try:
            collection.update_search_index(VECTOR_INDEX_NAME, vector_definition)
            print(f"Requested update of {VECTOR_INDEX_NAME}")
        except Exception as e:
            print(f"Vector index update note: {e}")
    else:
        try:
            collection.create_search_index(SearchIndexModel(
                definition=vector_definition,
                name=VECTOR_INDEX_NAME,
                type="vectorSearch",
            ))
            print(f"Requested creation of {VECTOR_INDEX_NAME}")
        except Exception as e:
            if existing_vector_name:
                VECTOR_INDEX_NAME = existing_vector_name
                print(f"Using existing vector index due to create limit: {VECTOR_INDEX_NAME}")
            else:
                print(f"Vector index create note: {e}")

    # Text index via PyMongo for keyword retrieval.
    text_definition = {
        "mappings": {
            "dynamic": False,
            "fields": {
                "content": {"type": "string", "analyzer": "lucene.standard"}
            }
        }
    }

    if TEXT_INDEX_NAME in existing_indexes:
        try:
            collection.update_search_index(TEXT_INDEX_NAME, text_definition)
            print(f"Requested update of {TEXT_INDEX_NAME}")
        except Exception as e:
            print(f"Text index update note: {e}")
    else:
        try:
            collection.create_search_index(SearchIndexModel(
                definition=text_definition,
                name=TEXT_INDEX_NAME,
                type="search",
            ))
            print(f"Requested creation of {TEXT_INDEX_NAME}")
        except Exception as e:
            if existing_text_name:
                TEXT_INDEX_NAME = existing_text_name
                print(f"Using existing text index due to create limit: {TEXT_INDEX_NAME}")
            else:
                print(f"Text index create note: {e}")

    # Wait briefly for Atlas indexes to become queryable before running search pipelines.
    max_wait_seconds = 45
    start = time.time()
    while time.time() - start < max_wait_seconds:
        statuses = {
            idx.get("name", "<unknown>"): idx.get("status", "UNKNOWN")
            for idx in collection.list_search_indexes()
            if idx.get("name") in {VECTOR_INDEX_NAME, TEXT_INDEX_NAME}
        }
        vector_ready = statuses.get(VECTOR_INDEX_NAME) in {"READY", None}
        text_ready = statuses.get(TEXT_INDEX_NAME) in {"READY", None}
        if vector_ready and text_ready:
            print("Atlas indexes are READY (or reused)")
            break
        time.sleep(3)
    else:
        print("Index build still in progress; later queries may need another minute.")

## Step 4: Validate Atlas Vector Search

Run a quick `$vectorSearch` smoke test before hybrid fusion to confirm the Atlas vector index is working.

In [ ]:
vector_smoke_ok = False

if collection is None:
    print("Skipping Atlas vector smoke test: collection is not available.")
else:
    try:
        probe_vector = embeddings.embed_query("What is reciprocal rank fusion?")
        smoke = list(collection.aggregate([
            {
                "$vectorSearch": {
                    "index": VECTOR_INDEX_NAME,
                    "path": "embedding",
                    "queryVector": probe_vector,
                    "numCandidates": 20,
                    "limit": 2,
                }
            },
            {"$project": {"_id": 0, "doc_id": 1, "content": 1}},
        ]))
        vector_smoke_ok = True
        print(f"Atlas vector smoke test OK: {len(smoke)} hit(s)")
        for item in smoke:
            print(item)
    except Exception as e:
        print(f"Atlas vector smoke test failed: {e}")

## Atlas Troubleshooting Guide (Workshop Quick Fixes)

Use this section when Atlas search stages fail or return unexpected empty results.

### Issue A: `$rankFusion` not allowed

- Symptom: `OperationFailure: $rankFusion is not allowed or the syntax is incorrect`
- Likely cause: your Atlas cluster or Search version does not support `$rankFusion` yet.
- Atlas fix: upgrade Atlas tier/version to one that supports `$rankFusion`, and ensure Search is enabled for the cluster.
- Notebook behavior: this notebook automatically falls back to manual RRF so the pipeline still runs.
- Quick verify: rerun the hybrid query step and confirm either (1) `$rankFusion` results are returned, or (2) fallback message + manual RRF results appear.

### Issue B: Maximum number of FTS indexes reached

- Symptom: `The maximum number of FTS indexes has been reached for this instance size.`
- Likely cause: Atlas Search index quota on current cluster tier.
- Atlas fix: delete unused Search/Vector indexes or scale cluster tier to increase index limits.
- Notebook behavior: index-create step now tries to reuse existing compatible indexes when possible.
- Quick verify: in Atlas UI, confirm one usable `vectorSearch` index and one usable `search` index exist for this collection.

### Issue C: Voyage client unavailable on Python 3.14

- Symptom: Voyage import/runtime errors from pydantic compatibility.
- Likely cause: current `voyageai` dependency stack is not fully compatible with Python 3.14.
- Atlas fix: none (client-side dependency issue).
- Notebook behavior: deterministic local embedding fallback is used, so Atlas Vector Search setup and test still execute.
- Quick verify: setup cell prints `Voyage client active: False` and subsequent Atlas steps still complete.

### Recommended Atlas Setup For This Workshop

- Use a cluster tier with Atlas Search + Vector Search enabled.
- Keep at least 2 Search indexes available per demo collection: one `vectorSearch`, one `search`.
- Ensure Search indexes are `READY` before running retrieval cells.
- If you need strict parity with lecture output, run notebooks on Python 3.11 or 3.12 for native Voyage client support.

## Step 5: Run hybrid search with `$rankFusion`

MongoDB Atlas runs both pipelines in parallel:

- `$vectorSearch` for semantic similarity
- `$search` for keyword matching
- `$rankFusion` to merge both rankings natively

In [ ]:
fused_results = []
query = "Which algorithm merges ranked lists without comparing raw scores?"

if collection is None:
    print("Skipping hybrid query: data source is missing.")
else:
    query_vector = embeddings.embed_query(query)

    pipeline = [
        {
            "$rankFusion": {
                "input": {
                    "pipelines": {
                        "vectorPipeline": [
                            {
                                "$vectorSearch": {
                                    "index": VECTOR_INDEX_NAME,
                                    "path": "embedding",
                                    "queryVector": query_vector,
                                    "numCandidates": 50,
                                    "limit": 10,
                                }
                            }
                        ],
                        "keywordPipeline": [
                            {
                                "$search": {
                                    "index": TEXT_INDEX_NAME,
                                    "text": {"query": query, "path": "content"},
                                }
                            },
                            {"$limit": 10},
                        ],
                    }
                },
                "combination": {
                    "weights": {
                        "vectorPipeline": 1,
                        "keywordPipeline": 1,
                    }
                }
            }
        },
        {"$limit": 10},
        {
            "$project": {
                "_id": 0,
                "doc_id": {"$ifNull": ["$doc_id", {"$toString": "$_id"}]},
                "content": {"$ifNull": ["$content", "$text"]},
            }
        },
    ]

    try:
        fused_results = list(collection.aggregate(pipeline))
        print(f"Returned {len(fused_results)} results from $rankFusion")
    except Exception as e:
        print(f"$rankFusion unavailable, falling back to manual RRF: {e}")

        keyword_results = list(collection.aggregate([
            {
                "$search": {
                    "index": TEXT_INDEX_NAME,
                    "text": {"query": query, "path": "content"},
                }
            },
            {"$limit": 10},
            {"$project": {"_id": 0, "doc_id": 1, "content": 1}},
        ]))

        vector_results = list(collection.aggregate([
            {
                "$vectorSearch": {
                    "index": VECTOR_INDEX_NAME,
                    "path": "embedding",
                    "queryVector": query_vector,
                    "numCandidates": 50,
                    "limit": 10,
                }
            },
            {"$project": {"_id": 0, "doc_id": 1, "content": 1}},
        ]))

        def reciprocal_rank_fusion(result_lists, k=60):
            scores = {}
            for ranked_list in result_lists:
                for rank, doc in enumerate(ranked_list, start=1):
                    doc_id = doc["doc_id"]
                    scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            return scores

        rrf_scores = reciprocal_rank_fusion([keyword_results, vector_results])
        doc_pool = {item["doc_id"]: item for item in keyword_results + vector_results}
        fused_results = sorted(
            doc_pool.values(),
            key=lambda item: rrf_scores.get(item["doc_id"], 0.0),
            reverse=True,
        )[:10]

        print(f"Returned {len(fused_results)} results from manual RRF fallback")

    for item in fused_results:
        print(item)

## Step 6: Re-rank with Voyage AI

Use Voyage AI `rerank-2.5` to improve the final ordering.

In [ ]:
if embeddings.client is None:
    print("Skipping re-ranking: Voyage client is not available in this environment.")
elif not fused_results:
    print("Skipping re-ranking: no fused results available.")
else:
    import voyageai

    vo = voyageai.Client(api_key=VOYAGE_API_KEY, base_url="https://ai.mongodb.com/v1/embeddings")
    rerank_result = vo.rerank(
        query=query,
        documents=[item["content"] for item in fused_results],
        model=RERANK_MODEL,
        top_k=3,
    )

    print("Top 3 after re-ranking:")
    for rank, item in enumerate(rerank_result.results, start=1):
        doc = fused_results[item.index]
        print(f"{rank}. {doc['doc_id']} | relevance={item.relevance_score:.4f}")
        print(f"   {doc['content']}")